In [37]:
import random

# Định nghĩa các hằng số dùng chung
GOAL = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
OPPOSITES = {'UP': 'DOWN', 'DOWN': 'UP', 'LEFT': 'RIGHT', 'RIGHT': 'LEFT'}

def is_solvable(state):
    """Đếm số lần đảo vị trí (inversions) gọn gàng bằng Generator để đảm bảo luôn giải được"""
    inversions = sum(
        1 for i in range(9) for j in range(i + 1, 9)
        if state[i] and state[j] and state[i] > state[j]
    )
    return inversions % 2 == 0

def generate_random_board():
    """Khởi tạo bàn cờ ngẫu nhiên hợp lệ"""
    while True:
        nums = list(range(9))
        random.shuffle(nums)
        if is_solvable(nums):
            return [nums[i:i + 3] for i in range(0, 9, 3)]

def find_blank(board):
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c

def get_legal_moves(board):
    r, c = find_blank(board)
    moves = {}
    if r > 0: moves['UP'] = (r - 1, c)
    if r < 2: moves['DOWN'] = (r + 1, c)
    if c > 0: moves['LEFT'] = (r, c - 1)
    if c < 2: moves['RIGHT'] = (r, c + 1)
    return moves

def apply_move(board, move):
    r, c = find_blank(board)
    nr, nc = get_legal_moves(board)[move]
    new_board = [row[:] for row in board]
    new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
    return new_board


In [38]:
def predict_score(board, visited_states):
    """[P1] Tính số ô sai vị trí. Cộng thêm điểm phạt cực nặng nếu trạng thái đã đi qua để tránh vòng lặp."""
    score = sum(
        1 for r in range(3) for c in range(3)
        if board[r][c] and board[r][c] != GOAL[r][c]
    )

    # Kỹ thuật tránh lặp: Phạt 100 điểm nếu AI định đi lại một trạng thái cũ
    if board in visited_states:
        score += 100

    return score

def print_board(board, title=None):
    """In ma trận cực ngắn gọn"""
    if title: 
        print(f"\n{title}")
    for row in board:
        print(" ".join(f"[{n if n else ' '}]" for n in row))


In [39]:
def solve_puzzle(max_steps=5000):
    board = generate_random_board()
    history = []
    visited_states = [board] # Khởi tạo bộ nhớ ghi lại các bàn cờ đã đi qua
    status, reason = "HALTED", "Quá số bước cho phép"

    print_board(board, "TRẠNG THÁI BẮT ĐẦU")

    for step in range(1, max_steps + 1):
        if board == GOAL:
            status, reason = "SOLVED", "Tuyệt vời! Đã tìm thấy đích"
            break

        legal_moves = get_legal_moves(board)

        # [P4] Chống đi lùi trực tiếp (ví dụ vừa đi UP thì không được đi DOWN ngay)
        if history:
            legal_moves.pop(OPPOSITES.get(history[-1]), None)

        # [P1] AI dự đoán nước đi tốt nhất dựa trên điểm số và bộ nhớ
        best_move = min(
            legal_moves, 
            key=lambda m: predict_score(apply_move(board, m), visited_states)
        )

        # Cập nhật trạng thái và lưu vào bộ nhớ
        history.append(best_move)
        board = apply_move(board, best_move)
        visited_states.append(board) # Ghi nhớ hình dáng bàn cờ này
        
        print_board(board, f"=> Bước {step}: Đi [{best_move}]")

    # Bảng tổng kết hiển thị
    print("\n" + "=" * 30)
    print("KẾT QUẢ TỔNG QUAN")
    print(f"Tổng số bước: {len(history)}")
    print(f"Trạng thái: {status} ({reason})")

# Khởi chạy AI
solve_puzzle()



TRẠNG THÁI BẮT ĐẦU
[3] [4] [8]
[2] [7] [6]
[ ] [5] [1]

=> Bước 1: Đi [UP]
[3] [4] [8]
[ ] [7] [6]
[2] [5] [1]

=> Bước 2: Đi [UP]
[ ] [4] [8]
[3] [7] [6]
[2] [5] [1]

=> Bước 3: Đi [RIGHT]
[4] [ ] [8]
[3] [7] [6]
[2] [5] [1]

=> Bước 4: Đi [DOWN]
[4] [7] [8]
[3] [ ] [6]
[2] [5] [1]

=> Bước 5: Đi [DOWN]
[4] [7] [8]
[3] [5] [6]
[2] [ ] [1]

=> Bước 6: Đi [LEFT]
[4] [7] [8]
[3] [5] [6]
[ ] [2] [1]

=> Bước 7: Đi [UP]
[4] [7] [8]
[ ] [5] [6]
[3] [2] [1]

=> Bước 8: Đi [UP]
[ ] [7] [8]
[4] [5] [6]
[3] [2] [1]

=> Bước 9: Đi [RIGHT]
[7] [ ] [8]
[4] [5] [6]
[3] [2] [1]

=> Bước 10: Đi [RIGHT]
[7] [8] [ ]
[4] [5] [6]
[3] [2] [1]

=> Bước 11: Đi [DOWN]
[7] [8] [6]
[4] [5] [ ]
[3] [2] [1]

=> Bước 12: Đi [DOWN]
[7] [8] [6]
[4] [5] [1]
[3] [2] [ ]

=> Bước 13: Đi [LEFT]
[7] [8] [6]
[4] [5] [1]
[3] [ ] [2]

=> Bước 14: Đi [LEFT]
[7] [8] [6]
[4] [5] [1]
[ ] [3] [2]

=> Bước 15: Đi [UP]
[7] [8] [6]
[ ] [5] [1]
[4] [3] [2]

=> Bước 16: Đi [UP]
[ ] [8] [6]
[7] [5] [1]
[4] [3] [2]

=> Bước 17: Đi [R